In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ---- TRANSFORMS ----
transform = transforms.ToTensor()

# ---- DATA ----
train_data = MNIST(root="data", train=True,  download=True, transform=transform)
test_data  = MNIST(root="data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)


class digitmodel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,10)


            
        )
    def forward(self,x):
        return self.net(x)
model = digitmodel()
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr = 0.001)
epochs = 1000
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for images, labels in train_loader:  # ← Bug 1 fixed!
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs,labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss:.2f}")
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs     = model(images)
        predictions = outputs.argmax(dim=1)
        correct    += (predictions == labels).sum().item()
        total      += labels.size(0)    # ← Bug 2 fixed!

accuracy = 100 * correct/total

print(f"Total accuracy: {accuracy:.2f}%")   # ← Bug 3 fixed!

# ---- SAVE ----
torch.save(model.state_dict(), "mnist.pth")
print("Model saved as mnist.pth")

# ---- VISUALIZE ONE PREDICTION ----
index = 69
image, true_label = test_data[index]

plt.imshow(image.squeeze(), cmap="gray")   # ← Bug 4 fixed!
plt.title(f"Actual label: {true_label}")
plt.axis("off")
plt.show()

# ---- PREDICT ONE IMAGE ----
image_flat = image.view(1, -1).to(device)  # ← Bug 5 fixed!

with torch.no_grad():
    output          = model(image_flat)
    predicted_label = output.argmax(dim=1).item()

print(f"User picked image index : {index}")
print(f"Actual label            : {true_label}")
print(f"Model prediction        : {predicted_label}")

RuntimeError: Error downloading train-images-idx3-ubyte.gz:
Tried https://ossci-datasets.s3.amazonaws.com/mnist/, got:
<urlopen error [Errno 11001] getaddrinfo failed>
Tried http://yann.lecun.com/exdb/mnist/, got:
<urlopen error [Errno 11001] getaddrinfo failed>


In [ ]:
## don't run on vs code locally